# Significance tests: seed-level paired t + per-user paired bootstrap

Two **complementary** tests measuring *different* sources of variance. A finding worth publishing must survive both.

1. **Seed-level paired t-test** — unit of variation = random training seed. Measures *training-stochasticity variance* (different init / shuffle / dropout mask ⇒ different learned model). Implemented in `evaluation.paired_significance`. Same seed ⇒ same data split on both sides, so we use the *paired* t-test (strictly more powerful than Welch's when pairing is valid). With n=5 seeds, df=4 — the test is conservative but exact under normality of the *differences*.
2. **Per-user paired bootstrap** — unit of variation = test user (thousands of them). Measures *test-population variance*: given one trained model, is the per-user metric diff reliable across users? Per-seed: score every test user with both checkpoints, compute per-user diff, resample users with replacement, report 95% CI on the mean diff. Aggregated across seeds with per-seed CI sign-agreement.

**Wilcoxon signed-rank** is also computed (`method='exact'`) for reference, but it is *not* part of the publishability filter: at n=5 its discrete null distribution has a hard two-sided p-floor of `2/2^5 = 0.0625`, so it can never reject at α=0.05 no matter how clean the data. It serves as a sanity check on the t-test's normality assumption.

A single-seed bootstrap would be over-confident — its CI is conditional on the trained model, hiding training-stochasticity variance. A multi-seed analysis without per-user bootstrap can miss whether the gain is reliable across the population. We need both.

Reuses existing checkpoints under `results/checkpoints/` — file names live in each eval JSON's `checkpoint` field. Per-user metrics are cached to parquet under `results/per_user/` so re-running is cheap.


In [1]:
import sys, os, json, math, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

warnings.filterwarnings('ignore', category=FutureWarning, module='recbole')

import numpy as np
import pandas as pd
import torch
from scipy import stats as sps

import config as cfg
from runner import _build_config, _instantiate_model, load_seed_result
from evaluation import paired_significance
from recbole.data import create_dataset, data_preparation

PROJECT = Path(cfg.PROJECT_ROOT)
EVAL_DIR = Path(cfg.EVAL_DIR)
CKPT_DIR = Path(cfg.CHECKPOINT_DIR)
PER_USER_DIR = PROJECT / 'results' / 'per_user'
PER_USER_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = cfg.DEVICE
print('device:', DEVICE)

2026-05-23 11:11:20.876466: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-23 11:11:20.946171: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-23 11:11:22.643209: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


device: cuda


/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. Seed-level paired test (uses existing helper)

In [2]:
DATASETS = ['ml-100k-iar', 'ml-1m', 'amazon-digital-music', 'amazon-office-products', 'steam-3k', 'steam-8k']
BASELINE = 'SASRec'
VARIANTS = ['IA-SASRec-Add', 'IA-SASRec-Mul', 'IA-SASRec-Val']
METRICS = ['ndcg@10', 'mrr@10', 'recall@10', 'ndcg@20', 'recall@20']

seed_rows = []
for ds in DATASETS:
    df = paired_significance(ds, BASELINE, VARIANTS, metrics=METRICS)
    if df.empty:
        continue
    df.insert(0, 'dataset', ds)
    seed_rows.append(df)
seed_df = pd.concat(seed_rows, ignore_index=True) if seed_rows else pd.DataFrame()

def _sig(p):
    if pd.isna(p): return ''
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.10: return '.'
    return ''

view = seed_df.copy()
view['t_sig'] = view['t_pvalue'].map(_sig)
view['w_sig'] = view['wilcoxon_pvalue'].map(_sig)
view[['dataset','model','metric','n','baseline_mean','challenger_mean','rel_diff_%','t_pvalue','t_sig','wilcoxon_pvalue','w_sig']]

/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
/home/coder/.pyenv/versions/3.12.0/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, corre

,dataset,model,metric,n,baseline_mean,challenger_mean,rel_diff_%,t_pvalue,t_sig,wilcoxon_pvalue,w_sig
0,ml-100k-iar,IA-SASRec-Add,ndcg@10,5,0.06220,0.05884,-5.401929,0.378174,,0.625000,
1,ml-100k-iar,IA-SASRec-Add,mrr@10,5,0.04128,0.03876,-6.104651,0.392869,,0.437500,
2,ml-100k-iar,IA-SASRec-Add,recall@10,5,0.13214,0.12642,-4.328742,0.431761,,0.589639,
3,ml-100k-iar,IA-SASRec-Add,ndcg@20,5,0.08646,0.08282,-4.210039,0.385626,,0.812500,
4,ml-100k-iar,IA-SASRec-Add,recall@20,5,0.22886,0.22206,-2.971249,0.518245,,0.812500,
...,...,...,...,...,...,...,...,...,...,...,...
70,steam,IA-SASRec-Val,ndcg@10,5,0.03134,0.03288,4.913848,0.340422,,0.437500,
71,steam,IA-SASRec-Val,mrr@10,5,0.02058,0.02290,11.273081,0.135040,,0.187500,
72,steam,IA-SASRec-Val,recall@10,5,0.06724,0.06574,-2.230815,0.505740,,0.625000,
73,steam,IA-SASRec-Val,ndcg@20,5,0.04214,0.04440,5.363075,0.169529,,0.187500,


## 2. Per-user metric computation

For each (dataset, model, seed) we:
1. Rebuild the RecBole `Config`/`Dataset`/dataloaders with the saved `best_params` and seed.
2. Load the checkpoint's `state_dict` into a fresh model.
3. For every test user: score all items, mask the user's history + padding, find the rank of the ground-truth target.
4. Derive per-user `recall@k`, `ndcg@k`, `mrr@k` (k ∈ {10, 20, 50, 100}).
5. Cache result to `results/per_user/<dataset>__<model>__seed<seed>.parquet`.

Note: this assumes leave-one-out (`LS: valid_and_test`) ⇒ exactly one ground-truth item per test user, which matches the project's eval config.

In [3]:
def _per_user_path(dataset: str, model: str, seed: int) -> Path:
    return PER_USER_DIR / f'{dataset}__{model}__seed{seed}.parquet'

K_LIST = [10, 20, 50, 100]

def compute_per_user(dataset: str, model: str, seed: int, force: bool = False) -> pd.DataFrame:
    """Return per-user metrics DataFrame; cache to parquet."""
    out = _per_user_path(dataset, model, seed)
    if out.exists() and not force:
        return pd.read_parquet(out)

    rec = load_seed_result(dataset, model, seed)
    best_params = dict(rec.get('best_params') or {})
    ckpt_name = rec.get('checkpoint')
    if not ckpt_name:
        raise FileNotFoundError(f'no checkpoint recorded for {dataset}/{model}/seed={seed}')
    ckpt_path = CKPT_DIR / ckpt_name
    if not ckpt_path.exists():
        raise FileNotFoundError(f'missing checkpoint file: {ckpt_path}')

    overrides = dict(best_params)
    overrides['seed'] = seed
    rb_config, model_cls = _build_config(dataset, model, overrides, epochs=1, saved=False)
    rb_dataset = create_dataset(rb_config)
    _train, _valid, test_data = data_preparation(rb_config, rb_dataset)
    net = _instantiate_model(rb_config, test_data._dataset, model_cls)

    state = torch.load(ckpt_path, map_location=DEVICE)
    sd = state.get('state_dict', state)
    net.load_state_dict(sd)
    net.eval()

    uid_field = rb_dataset.uid_field
    iid_field = rb_dataset.iid_field

    rows = []
    with torch.no_grad():
        for batch in test_data:
            # FullSort dataloaders yield (interaction, history, positive_u, positive_i)
            if isinstance(batch, tuple) and len(batch) == 4:
                interaction, history, positive_u, positive_i = batch
            else:
                # Some sequential dataloaders return only the interaction.
                interaction = batch
                history = None
                positive_u = torch.arange(len(interaction[uid_field]))
                positive_i = interaction[iid_field]

            interaction = interaction.to(DEVICE)
            scores = net.full_sort_predict(interaction)
            n_users = interaction[uid_field].shape[0]
            if scores.dim() == 1:
                scores = scores.view(n_users, -1)
            scores[:, 0] = -float('inf')  # padding

            if history is not None:
                history_u, history_i = history
                if history_u.numel() > 0:
                    scores[history_u.to(DEVICE), history_i.to(DEVICE)] = -float('inf')

            uid_tokens = rb_dataset.id2token(uid_field, interaction[uid_field].cpu().numpy())

            # In leave-one-out + full eval, positive_u indexes into the batch's user rows,
            # and positive_i is the held-out target item id.
            pu = positive_u.cpu().numpy()
            pi = positive_i.cpu().numpy()

            # Group ground-truth items per user (usually 1 per user; handle >=1).
            for row_idx in range(n_users):
                mask = pu == row_idx
                if not mask.any():
                    continue
                gt_items = pi[mask].astype(np.int64)
                row_scores = scores[row_idx]
                # rank = number of items with strictly greater score, +1
                gt_scores = row_scores[torch.as_tensor(gt_items, device=DEVICE)]
                # Best of the ground-truth items (smallest rank)
                gt_best_score = gt_scores.max().item()
                rank = int((row_scores > gt_best_score).sum().item()) + 1
                rec_row = {'user_id': str(uid_tokens[row_idx]), 'rank': rank}
                for k in K_LIST:
                    hit = 1.0 if rank <= k else 0.0
                    rec_row[f'recall@{k}'] = hit  # single ground truth ⇒ recall == hit
                    rec_row[f'hit@{k}'] = hit
                    rec_row[f'ndcg@{k}'] = (1.0 / math.log2(rank + 1)) if rank <= k else 0.0
                    rec_row[f'mrr@{k}'] = (1.0 / rank) if rank <= k else 0.0
                rows.append(rec_row)

    df = pd.DataFrame(rows)
    df.to_parquet(out, index=False)
    return df

# Quick sanity check on a single seed.
_demo = compute_per_user('amazon-office-products', 'SASRec', 2020)
print('users:', len(_demo))
print('aggregate NDCG@10 (per-user):', _demo['ndcg@10'].mean())
print('aggregate NDCG@10 (eval JSON):', load_seed_result('amazon-office-products','SASRec',2020)['test_result']['ndcg@10'])

users: 4905
aggregate NDCG@10 (per-user): 0.045783615621113466
aggregate NDCG@10 (eval JSON): 0.0458


**Sanity check above:** the per-user mean of `ndcg@10` should match the aggregate `ndcg@10` from the eval JSON (within float noise). If not, the masking or ground-truth extraction is off and the rest of this notebook is invalid — stop and debug.

In [4]:
# Compute per-user metrics for every (dataset, model, seed) that has a checkpoint.
TO_RUN = []
for ds in DATASETS:
    for m in [BASELINE] + VARIANTS:
        for f in sorted(EVAL_DIR.glob(f'{ds}__{m}__seed*.json')):
            seed = int(f.stem.rsplit('seed', 1)[1])
            try:
                rec = json.loads(f.read_text())
            except Exception:
                continue
            if not rec.get('checkpoint'):
                continue
            TO_RUN.append((ds, m, seed))

print(f'tasks: {len(TO_RUN)}')
for ds, m, s in TO_RUN:
    out = _per_user_path(ds, m, s)
    if out.exists():
        continue
    try:
        compute_per_user(ds, m, s)
        print(f'  ok   {ds}/{m}/seed={s}')
    except Exception as e:
        print(f'  FAIL {ds}/{m}/seed={s}: {e}')

tasks: 96
  ok   steam/SASRec/seed=2020
  ok   steam/SASRec/seed=2021
  ok   steam/SASRec/seed=2022
  ok   steam/SASRec/seed=2023
  ok   steam/SASRec/seed=2024
  ok   steam/IA-SASRec-Add/seed=2020
  ok   steam/IA-SASRec-Add/seed=2021
  ok   steam/IA-SASRec-Add/seed=2022
  ok   steam/IA-SASRec-Add/seed=2023
  ok   steam/IA-SASRec-Add/seed=2024
  ok   steam/IA-SASRec-Mul/seed=2020
  ok   steam/IA-SASRec-Mul/seed=2021
  ok   steam/IA-SASRec-Mul/seed=2022
  ok   steam/IA-SASRec-Mul/seed=2023
  ok   steam/IA-SASRec-Mul/seed=2024
  ok   steam/IA-SASRec-Val/seed=2020
  ok   steam/IA-SASRec-Val/seed=2021
  ok   steam/IA-SASRec-Val/seed=2022
  ok   steam/IA-SASRec-Val/seed=2023
  ok   steam/IA-SASRec-Val/seed=2024


## 3. Paired per-user bootstrap CIs + effect size

For each (dataset, variant, metric):

1. For every seed present in *both* baseline and variant, align users (inner-join on `user_id`) and compute per-user diff `d_u = variant(u) − baseline(u)`.
2. **Per-seed bootstrap (rigorous):** for each seed independently, resample users with replacement `B=2000` times, report the 95% CI on the mean diff. Then summarise across seeds: how many seeds have CI entirely > 0? entirely < 0?
3. **Pooled bootstrap (coarse summary):** concatenate diffs across seeds and bootstrap once. *Caveat:* this treats `(user_u, seed_2020)` and `(user_u, seed_2021)` as exchangeable, which they aren't — same user implies correlated errors. Use per-seed CIs as the rigorous reading; the pooled CI is for at-a-glance comparison only.
4. **Cohen's d:** `mean(d_u) / std(d_u)`, per seed and pooled. With thousands of test users a p-value becomes trivially small for tiny effects — d quantifies whether the gain is *practically* meaningful. Rules of thumb: |d| ≥ 0.2 small, ≥ 0.5 medium, ≥ 0.8 large. For top-k rec-sys gains, expect tiny d (often 0.02–0.05) even when p ≪ 0.05; that's the practical-significance reality check.
5. **Two-sided bootstrap p-value:** `2 * min(P(mean<=0), P(mean>=0))`, floored at `1/B`.

Note on the seed-variance question: bootstrap-resampling users does not account for training instability across seeds. With n=3–5 seeds, a nested "resample seeds then users" bootstrap is degenerate (the outer resample has too few distinct multisets). The honest treatment is to report the seed-level paired test (§1) *alongside* the per-user bootstrap — they answer different questions and a finding worth publishing should survive both.

In [5]:
def _bootstrap_ci(diffs: np.ndarray, n_boot: int, rng: np.random.Generator) -> tuple[float, float, float]:
    """Return (ci_lo, ci_hi, two_sided_p) from a paired-diff bootstrap.

    Uses percentile CI and the proportion-of-resamples definition of p,
    floored at 1/n_boot to avoid p=0.
    """
    n = len(diffs)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diffs[idx].mean(axis=1)
    lo, hi = np.quantile(boot_means, [0.025, 0.975])
    n_le = int((boot_means <= 0).sum())
    n_ge = int((boot_means >= 0).sum())
    p = 2.0 * min(n_le, n_ge) / n_boot
    return float(lo), float(hi), max(p, 1.0 / n_boot)


def _cohens_d(diffs: np.ndarray) -> float:
    """Cohen's d on paired diffs: mean / std (unbiased)."""
    if len(diffs) < 2:
        return float('nan')
    s = diffs.std(ddof=1)
    return float(diffs.mean() / s) if s > 0 else float('nan')


def paired_user_bootstrap(
    dataset: str,
    variant: str,
    metric: str,
    n_boot: int = 2000,
    rng: np.random.Generator | None = None,
) -> dict:
    rng = rng or np.random.default_rng(0)
    base_files = sorted(PER_USER_DIR.glob(f'{dataset}__{BASELINE}__seed*.parquet'))
    base_seeds = {int(p.stem.rsplit('seed', 1)[1]): p for p in base_files}
    var_files = sorted(PER_USER_DIR.glob(f'{dataset}__{variant}__seed*.parquet'))
    var_seeds = {int(p.stem.rsplit('seed', 1)[1]): p for p in var_files}
    shared = sorted(set(base_seeds) & set(var_seeds))
    if not shared:
        return {}

    # Per-seed: bootstrap independently, Cohen's d, store.
    per_seed = []
    pooled_diffs = []
    pooled_base_means = []
    for s in shared:
        b = pd.read_parquet(base_seeds[s], columns=['user_id', metric]).rename(columns={metric: 'b'})
        c = pd.read_parquet(var_seeds[s], columns=['user_id', metric]).rename(columns={metric: 'c'})
        m = b.merge(c, on='user_id', how='inner')
        d = (m['c'] - m['b']).to_numpy()
        pooled_diffs.append(d)
        pooled_base_means.append(m['b'].mean())
        lo, hi, p = _bootstrap_ci(d, n_boot, rng)
        per_seed.append({
            'seed': s,
            'n_users': len(d),
            'mean_diff': float(d.mean()),
            'ci_lo': lo,
            'ci_hi': hi,
            'p_boot': p,
            'cohens_d': _cohens_d(d),
            'sig_pos': lo > 0,
            'sig_neg': hi < 0,
        })

    diffs_pool = np.concatenate(pooled_diffs)
    base_mean = float(np.mean(pooled_base_means))
    lo_p, hi_p, p_pool = _bootstrap_ci(diffs_pool, n_boot, rng)
    obs_mean = float(diffs_pool.mean())
    n_seeds_sig_pos = sum(1 for r in per_seed if r['sig_pos'])
    n_seeds_sig_neg = sum(1 for r in per_seed if r['sig_neg'])

    return {
        'dataset': dataset,
        'model': variant,
        'metric': metric,
        'n_seeds': len(shared),
        'n_users_total': len(diffs_pool),
        'baseline_mean': base_mean,
        'mean_diff': obs_mean,
        'rel_diff_%': (obs_mean / base_mean * 100.0) if base_mean else 0.0,
        'ci_lo_pool': lo_p,
        'ci_hi_pool': hi_p,
        'p_boot_pool': p_pool,
        'cohens_d_pool': _cohens_d(diffs_pool),
        'seeds_sig_pos': n_seeds_sig_pos,
        'seeds_sig_neg': n_seeds_sig_neg,
        'per_seed': per_seed,
    }


boot_rows = []
for ds in DATASETS:
    for v in VARIANTS:
        for met in METRICS:
            r = paired_user_bootstrap(ds, v, met)
            if r:
                boot_rows.append(r)
boot_df = pd.DataFrame(boot_rows)
boot_df['sig_pool'] = boot_df['p_boot_pool'].map(_sig)
boot_df['seeds_sig'] = boot_df.apply(lambda r: f"+{int(r['seeds_sig_pos'])}/-{int(r['seeds_sig_neg'])}/{int(r['n_seeds'])}", axis=1)
boot_df[['dataset','model','metric','n_seeds','n_users_total','baseline_mean','mean_diff','rel_diff_%','ci_lo_pool','ci_hi_pool','p_boot_pool','sig_pool','cohens_d_pool','seeds_sig']]

,dataset,model,metric,n_seeds,n_users_total,baseline_mean,mean_diff,rel_diff_%,ci_lo_pool,ci_hi_pool,p_boot_pool,sig_pool,cohens_d_pool,seeds_sig
0,ml-100k-iar,IA-SASRec-Add,ndcg@10,5,4715,0.062207,-0.003385,-5.441519,-0.007972,0.000916,0.115,,-0.020835,+0/-1/5
1,ml-100k-iar,IA-SASRec-Add,mrr@10,5,4715,0.041266,-0.002521,-6.109771,-0.006569,0.001292,0.195,,-0.017634,+0/-1/5
2,ml-100k-iar,IA-SASRec-Add,recall@10,5,4715,0.132131,-0.005726,-4.333868,-0.014634,0.003181,0.217,,-0.017930,+0/-0/5
3,ml-100k-iar,IA-SASRec-Add,ndcg@20,5,4715,0.086461,-0.003652,-4.223870,-0.008212,0.000443,0.085,.,-0.023821,+0/-1/5
4,ml-100k-iar,IA-SASRec-Add,recall@20,5,4715,0.228844,-0.006787,-2.965709,-0.016972,0.002975,0.192,,-0.018748,+0/-1/5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,steam,IA-SASRec-Val,ndcg@10,5,12210,0.031340,0.001497,4.776613,-0.000331,0.003331,0.097,.,0.014113,+1/-0/5
71,steam,IA-SASRec-Val,mrr@10,5,12210,0.020580,0.002329,11.316479,0.000757,0.004004,0.004,**,0.024871,+2/-0/5
72,steam,IA-SASRec-Val,recall@10,5,12210,0.067240,-0.001474,-2.192448,-0.005078,0.002293,0.453,,-0.006896,+0/-0/5
73,steam,IA-SASRec-Val,ndcg@20,5,12210,0.042161,0.002250,5.337873,0.000521,0.004016,0.005,**,0.022366,+2/-0/5


## 4. Combined view

In [6]:
left = seed_df[['dataset','model','metric','n','rel_diff_%','t_pvalue','wilcoxon_pvalue']].rename(
    columns={'n':'n_seeds','rel_diff_%':'rel_diff_seed_%'}
)
right = boot_df[[
    'dataset','model','metric','n_users_total','rel_diff_%',
    'ci_lo_pool','ci_hi_pool','p_boot_pool','cohens_d_pool',
    'seeds_sig_pos','seeds_sig_neg',
]].rename(columns={'rel_diff_%':'rel_diff_user_%'})
combined = left.merge(right, on=['dataset','model','metric'], how='outer')
combined['t_sig'] = combined['t_pvalue'].map(_sig)
combined['w_sig'] = combined['wilcoxon_pvalue'].map(_sig)
combined['boot_sig'] = combined['p_boot_pool'].map(_sig)
combined['seeds_sig'] = combined.apply(
    lambda r: f"+{int(r['seeds_sig_pos'])}/-{int(r['seeds_sig_neg'])}/{int(r['n_seeds'])}"
    if pd.notna(r.get('seeds_sig_pos')) else '', axis=1
)
combined = combined.sort_values(['dataset','model','metric']).reset_index(drop=True)
combined

,dataset,model,metric,n_seeds,rel_diff_seed_%,t_pvalue,wilcoxon_pvalue,n_users_total,rel_diff_user_%,ci_lo_pool,ci_hi_pool,p_boot_pool,cohens_d_pool,seeds_sig_pos,seeds_sig_neg,t_sig,w_sig,boot_sig,seeds_sig
0,amazon-digital-music,IA-SASRec-Add,mrr@10,5,1.776462,0.082568,0.1250,27705,1.793676,-0.000136,0.002109,0.0980,0.009736,0,0,.,,.,+0/-0/5
1,amazon-digital-music,IA-SASRec-Add,ndcg@10,5,-1.161963,0.243204,0.4375,27705,-1.137133,-0.002356,0.000507,0.1760,-0.007894,0,1,,,,+0/-1/5
2,amazon-digital-music,IA-SASRec-Add,ndcg@20,5,-2.125604,0.013918,0.0625,27705,-2.121175,-0.003450,-0.000880,0.0020,-0.018894,0,1,*,.,**,+0/-1/5
3,amazon-digital-music,IA-SASRec-Add,recall@10,5,-3.877239,0.029973,0.1250,27705,-3.863592,-0.010323,-0.003754,0.0005,-0.025915,0,3,*,,**,+0/-3/5
4,amazon-digital-music,IA-SASRec-Add,recall@20,5,-4.553172,0.008647,0.0625,27705,-4.554318,-0.015233,-0.008157,0.0005,-0.038198,0,4,**,.,**,+0/-4/5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,steam,IA-SASRec-Val,mrr@10,5,11.273081,0.135040,0.1875,12210,11.316479,0.000757,0.004004,0.0040,0.024871,2,0,,,**,+2/-0/5
71,steam,IA-SASRec-Val,ndcg@10,5,4.913848,0.340422,0.4375,12210,4.776613,-0.000331,0.003331,0.0970,0.014113,1,0,,,.,+1/-0/5
72,steam,IA-SASRec-Val,ndcg@20,5,5.363075,0.169529,0.1875,12210,5.337873,0.000521,0.004016,0.0050,0.022366,2,0,,,**,+2/-0/5
73,steam,IA-SASRec-Val,recall@10,5,-2.230815,0.505740,0.6250,12210,-2.192448,-0.005078,0.002293,0.4530,-0.006896,0,0,,,,+0/-0/5


In [7]:
# "Publishable" rows must satisfy ALL three independent criteria:
#   1. Seed-level paired t-test rejects (training-stochasticity variance).
#   2. Pooled per-user bootstrap rejects (test-population variance).
#   3. Majority of per-seed CIs agree on sign (guards against one lucky seed driving the pool).
# Wilcoxon is informational only — at n=5 its exact p-floor is 0.0625, so it cannot reject at α=0.05.
def _seed_majority(r):
    if pd.isna(r.get('n_seeds')):
        return False
    n = int(r['n_seeds'])
    return int(r['seeds_sig_pos']) > n/2 or int(r['seeds_sig_neg']) > n/2

robust = combined[
    (combined['t_pvalue'] < 0.05)
    & (combined['p_boot_pool'] < 0.05)
    & combined.apply(_seed_majority, axis=1)
].copy()
robust[[
    'dataset','model','metric',
    'rel_diff_seed_%','rel_diff_user_%',
    't_pvalue','wilcoxon_pvalue',
    'ci_lo_pool','ci_hi_pool','p_boot_pool',
    'cohens_d_pool','seeds_sig',
]]

,dataset,model,metric,rel_diff_seed_%,rel_diff_user_%,t_pvalue,wilcoxon_pvalue,ci_lo_pool,ci_hi_pool,p_boot_pool,cohens_d_pool,seeds_sig
3,amazon-digital-music,IA-SASRec-Add,recall@10,-3.877239,-3.863592,0.029973,0.1250,-0.010323,-0.003754,0.0005,-0.025915,+0/-3/5
4,amazon-digital-music,IA-SASRec-Add,recall@20,-4.553172,-4.554318,0.008647,0.0625,-0.015233,-0.008157,0.0005,-0.038198,+0/-4/5
7,amazon-digital-music,IA-SASRec-Mul,ndcg@20,-3.265700,-3.271248,0.006106,0.0625,-0.004637,-0.002028,0.0005,-0.030466,+0/-3/5
9,amazon-digital-music,IA-SASRec-Mul,recall@20,-3.750579,-3.746518,0.006651,0.0625,-0.013211,-0.006100,0.0005,-0.032216,+0/-4/5
29,amazon-office-products,IA-SASRec-Val,recall@20,-7.155892,-7.148552,0.018877,0.0625,-0.014557,-0.007624,0.0005,-0.040644,+0/-4/5
62,steam,IA-SASRec-Add,ndcg@20,5.363075,5.270067,0.034702,0.0625,0.001036,0.003419,0.0005,0.032832,+3/-0/5


## 5. λ calibration plot — the mechanism figure

Each IA-SASRec variant has one learnable scalar λ per attention layer, initialised to 1.0. At λ=0 the variant collapses to vanilla SASRec. Variants log per-layer λ to `results/eval/*.json` under `intensity_params`. We plot per-(dataset, variant) mean λ against the seed-mean test-NDCG@10 delta vs. SASRec.

If the architecture works as designed:
- large λ ⇔ model uses intensity ⇔ measurable effect on metric (positive or negative)
- small λ ⇔ model opts out ⇔ result ≈ SASRec

Steam should cluster top-right (high λ, positive Δ); ml-1m Mul/Val should sit near the origin (low λ, no Δ).

In [ ]:
import json, glob
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

rows = []
for ds in DATASETS:
    base_files = sorted(glob.glob(f'../results/eval/{ds}__SASRec__seed*.json'))
    if not base_files:
        continue
    base_ndcg = [json.load(open(f))['test_result']['ndcg@10'] for f in base_files]
    base_mean = sum(base_ndcg) / len(base_ndcg)
    for v in VARIANTS:
        files = sorted(glob.glob(f'../results/eval/{ds}__{v}__seed*.json'))
        if not files:
            continue
        ndcgs, lambdas = [], []
        for f in files:
            d = json.load(open(f))
            ndcgs.append(d['test_result']['ndcg@10'])
            ip = d.get('intensity_params') or {}
            ll = [val for k, val in ip.items() if 'lambda' in k]
            if ll:
                lambdas.append(sum(ll) / len(ll))
        if not ndcgs or not lambdas:
            continue
        rows.append({
            'dataset': ds, 'variant': v,
            'lambda_mean': sum(lambdas)/len(lambdas),
            'rel_delta': (sum(ndcgs)/len(ndcgs) - base_mean)/base_mean * 100,
            'n_seeds': len(ndcgs),
        })

lam_df = pd.DataFrame(rows)
print(lam_df.to_string(index=False))

markers = {'IA-SASRec-Add': 'o', 'IA-SASRec-Mul': 's', 'IA-SASRec-Val': '^'}
palette = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown']
colors = {ds: c for ds, c in zip(DATASETS, palette)}

fig, ax = plt.subplots(figsize=(10, 6))
ymax = max(lam_df['rel_delta'].max() + 2, 8)
ymin = min(lam_df['rel_delta'].min() - 2, -10)
ax.set_ylim(ymin, ymax); ax.set_xlim(0, 1.05)
ax.axhspan(0, ymax, alpha=0.05, color='green', zorder=0)
ax.axhspan(ymin, 0, alpha=0.05, color='red', zorder=0)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)

for _, r in lam_df.iterrows():
    ax.scatter(r['lambda_mean'], r['rel_delta'],
               marker=markers[r['variant']], color=colors[r['dataset']],
               s=220, edgecolor='black', linewidth=0.8, alpha=0.9, zorder=3)

ax.set_xlabel('Mean learned $\\lambda$ (across layers, across seeds; init=1.0)', fontsize=11)
ax.set_ylabel('Relative $\\Delta$ NDCG@10 vs. SASRec (%)', fontsize=11)
ax.set_title('IA-SASRec: $\\lambda$-gate calibration vs. metric gain', fontsize=12)
ax.text(0.99, ymax-0.4, 'high $\\lambda$ · gain', ha='right', va='top',
        fontsize=9, color='gray', style='italic')
ax.text(0.99, ymin+0.4, 'high $\\lambda$ · loss\n(saturation / noise)', ha='right', va='bottom',
        fontsize=9, color='gray', style='italic')
ax.text(0.02, -0.3, 'low $\\lambda$ · model opts out', ha='left', va='top',
        fontsize=9, color='gray', style='italic')

variant_h = [Line2D([0],[0], marker=m, color='w', markerfacecolor='gray',
                    markeredgecolor='black', markersize=12,
                    label=v.replace('IA-SASRec-','IA-'))
             for v, m in markers.items()]
dataset_h = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                    markeredgecolor='black', markersize=12,
                    label=ds.replace('amazon-','am-'))
             for ds, c in colors.items() if ds in lam_df['dataset'].unique()]
leg1 = ax.legend(handles=variant_h, loc='upper left', bbox_to_anchor=(1.02, 1.0),
                 title='Variant', fontsize=9, framealpha=0.95)
ax.add_artist(leg1)
ax.legend(handles=dataset_h, loc='upper left', bbox_to_anchor=(1.02, 0.62),
          title='Dataset', fontsize=9, framealpha=0.95)
plt.tight_layout()
plt.savefig('../results/lambda_vs_delta.png', dpi=140, bbox_inches='tight')
plt.show()
